In [9]:
import pandas as pd
from pathlib import Path
from matplotlib import pyplot as plt
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [10]:
year = 2021

In [11]:
data_dir = Path("data")
INDIR = Path(f"../data/data_processed/{year}")
OUTDIR_IMG = Path(f"../report/img/{year}")
OUTDIR_IMG.mkdir(parents=True, exist_ok=True)

In [12]:
city_file = INDIR / f"ENEM_SCORES_MUNICIPALITIES_BRAZIL_PROCESSED_{year}.csv"
df = pd.read_csv(city_file, sep=",")

In [13]:
df.head()

,CITY_CODE,STATE,NUM_PARTICIPANTS,NATURAL_SCIENCES_SCORE_AVG,HUMANITIES_SCORE_AVG,LANGUAGES_SCORE_AVG,MATH_SCORE_AVG,ESSAY_SCORE_AVG,OVERALL_SCORE_AVG,CITY,FAMILY_INCOME_SM_AVG
0,1100015,RO,180,481.651111,505.165556,483.506111,507.848889,629.777778,521.589889,ALTA FLORESTA D'OESTE,1.732490
1,1100023,RO,1068,475.760955,502.431929,487.627809,514.135019,609.906367,517.972416,ARIQUEMES,1.949351
2,1100049,RO,1289,489.785881,515.289139,497.296664,528.150737,633.374709,532.779426,CACOAL,2.156101
3,1100056,RO,204,473.096569,497.325490,478.176471,496.488725,613.039216,511.625294,CEREJEIRAS,1.941948
4,1100064,RO,173,484.414451,515.620231,491.889595,534.378035,612.254335,527.711329,COLORADO DO OESTE,2.332969


In [ ]:
score_columns = [
    'NATURAL_SCIENCES_SCORE_AVG',
    'HUMANITIES_SCORE_AVG',
    'LANGUAGES_SCORE_AVG',
    'MATH_SCORE_AVG',
    'ESSAY_SCORE_AVG',
]

subject_names = {
    'NATURAL_SCIENCES_SCORE_AVG': 'Natural Sciences and its Technologies',
    'MATH_SCORE_AVG': 'Mathematics and its Technologies',
    'HUMANITIES_SCORE_AVG': 'Humanities and its Technologies',
    'LANGUAGES_SCORE_AVG': 'Languages, Codes and its Technologies',
    'ESSAY_SCORE_AVG': 'Essay',
}

# Okabe-Ito palette (color-blind friendly)
color_map = {
    'NATURAL_SCIENCES_SCORE_AVG': '#0072B2',  # blue
    'MATH_SCORE_AVG': '#E69F00',              # orange
    'HUMANITIES_SCORE_AVG': '#009E73',        # green
    'LANGUAGES_SCORE_AVG': '#D55E00',         # orange-red
    'ESSAY_SCORE_AVG': '#CC79A7',             # purple
}

for col in score_columns:
    area_name = subject_names.get(col, col)
    area_color = color_map.get(col, '#0072B2')

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=('Histogram', 'Box Plot')
    )

    fig.add_trace(
        go.Histogram(
            x=df[col].round(2),
            nbinsx=200,
            histnorm='probability density',
            marker_color=area_color,
            name=area_name
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Box(y=df[col], marker_color=area_color, name=area_name),
        row=1, col=2
    )

    fig.update_layout(
        title_text=f'Score Distribution - {area_name}',
        showlegend=False,
        width=1200,
        height=450,
        template='plotly_white'
    )
    fig.show()

In [15]:
col_income = 'FAMILY_INCOME_SM_AVG'
income_2c = df[col_income].round(2)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=('Histogram', 'Box Plot')
)

fig.add_trace(
    go.Histogram(
        x=income_2c,
        nbinsx=200,
        histnorm='probability density',
        marker_color='#0072B2',
        name='Average Family Income (MW)'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Box(
        y=income_2c,
        marker_color='#0072B2',
        name='Average Family Income (MW)'
    ),
    row=1, col=2
)

fig.update_layout(
    title_text='Average Family Income Distribution (MW)',
    showlegend=False,
    width=1200,
    height=450,
    template='plotly_white'
)

fig.show()

In [ ]:
candidates_income = ["FAMILY_INCOME_SM_AVG"]
candidates_score = ["OVERALL_SCORE_AVG"]
candidates_municipality = ["NO_MUNICIPIO", "CITY", "NOME_MUNICIPIO"]
candidates_state = ["SG_UF", "STATE", "SIGLA_UF"]

col_municipality = next((c for c in candidates_municipality if c in df.columns), None)
col_state = next((c for c in candidates_state if c in df.columns), None)

if col_municipality is None or col_state is None:
    raise KeyError(
        "Municipality/State columns not found. Available: "
        f"{list(df.columns)}"
    )

df_plot = df[[
    candidates_income[0],
    candidates_score[0],
    col_municipality,
    col_state,
]].dropna()

correlation = df_plot[candidates_income[0]].corr(df_plot[candidates_score[0]])

fig = go.Figure(
    data=go.Scatter(
        x=df_plot[candidates_income[0]],
        y=df_plot[candidates_score[0]],
        mode="markers",
        marker=dict(color="#0072B2", size=6),
        customdata=df_plot[[col_municipality, col_state]].values,
        hovertemplate=(
            "Municipality Name: %{customdata[0]}<br>"
            "State: %{customdata[1]}<br>"
            "Average Overall Score: %{y:.2f}<br>"
            "Average Family Income (MW): %{x:.2f}<extra></extra>"
        ),
        name="Municipalities"
    )
)

fig.update_layout(
    title=f"Scatter: Family Income vs. Average Overall Score (r = {correlation:.3f})",
    xaxis_title="Average Family Income (MW)",
    yaxis_title="Average Overall Score",
    template="plotly_white",
    width=1200,
    height=450,
)

fig.show()